In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-17 20:50:14,900 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b
2026-04-17 20:50:15,051 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: openai


In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-17 20:50:18,448 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-17 20:50:18,449 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-17 20:50:18,449 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-17 20:50:18,450 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [ ]:
test_invoke_with_tool(agent)

In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [4]:
await test_astream_with_tool(agent)

2026-04-17 20:50:22,229 | INFO | 对话历史已清空


round 1


2026-04-17 20:50:22,497 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



thinking content:
用户要求我：
1. 使用工具翻译中文"你是谁，在哪里"到英语
2. 使用计算器计算 3^22

我需要：
1. 调用 translate_tool 翻译文本
2. 调用 calculator 计算 3^22

这两个工具调用可以并行执行，因为它们互不依赖。

content:



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3^22'}


2026-04-17 20:50:24,712 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"



round 2

thinking content:
关于翻译工具的结果，返回的是原文"你是谁，在哪里"，而不是英语翻译。这说明翻译工具可能存在问题或返回格式有误。让我检查一下翻译工具是否正常工作。

看起来翻译工具没有正确执行翻译，返回了原文。我应该指出这个问题。

计算结果是正确的：3^22 = 31,381,059,609

让我给用户一个清晰的答复。

content:


## 翻译工具问题

**翻译工具存在异常**：返回的结果是原文"你是谁，在哪里"，而不是英语翻译。这看起来像是工具未能正确执行翻译任务。

如果您需要，我可以尝试其他方式翻译这句话：
- "你是谁，在哪里" 的正确英语翻译应为：**"Who are you, where are you?"**

## 计算结果

**3^22 = 31,381,059,609**（约 313 亿）

这个计算结果是正确的。
final res:


## 翻译工具问题

**翻译工具存在异常**：返回的结果是原文"你是谁，在哪里"，而不是英语翻译。这看起来像是工具未能正确执行翻译任务。

如果您需要，我可以尝试其他方式翻译这句话：
- "你是谁，在哪里" 的正确英语翻译应为：**"Who are you, where are you?"**

## 计算结果

**3^22 = 31,381,059,609**（约 313 亿）

这个计算结果是正确的。


In [ ]:
raw_history=agent.get_raw_history()  


In [ ]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history==raw_history2

In [ ]:
await agent.astream_invoke(f"我们刚才聊了什么")


In [ ]:
llm= EasyLLM(provider="anthropic_native",base_url="http://127.0.0.1:5124",api_key="122",model="qwen3.5-9b")
agent.change_model(llm=llm)

2026-04-17 20:48:30,324 | INFO | EasyLLM 初始化完成: provider=anthropic_native, model=qwen3.5-9b


In [11]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

False

In [14]:
await agent.astream_invoke(f"我们刚才聊了什么")


2026-04-17 20:48:33,124 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/messages "HTTP/1.1 200 OK"


round 1

thinking content:
用户问我们刚才聊了什么，我可以直接回答，不需要使用工具。让我总结一下之前的对话内容。

content:


我们刚才聊了以下内容：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 翻译工具出现了问题，未能正确翻译
   - 正确翻译应为："Who are you, where are you"

2. **计算任务**：计算 3^22
   - 结果：31,381,059,609
final res:


我们刚才聊了以下内容：

1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语
   - 翻译工具出现了问题，未能正确翻译
   - 正确翻译应为："Who are you, where are you"

2. **计算任务**：计算 3^22
   - 结果：31,381,059,609


/home/wxd/miniconda3/envs/llm/lib/python3.10/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ParsedTextBlock[~ResponseFormatT]` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `RedactedThinkingBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', parsed_output=None), input_type=ParsedTextBlock])
  PydanticSerializationUnexpectedValue(Expected `ToolUseBlock` - serialized value may not be as expected [field_name='content', input_value=ParsedTextBlock(citations...xt', par

'\n\n我们刚才聊了以下内容：\n\n1. **翻译任务**：将中文"你是谁，在哪里"翻译成英语\n   - 翻译工具出现了问题，未能正确翻译\n   - 正确翻译应为："Who are you, where are you"\n\n2. **计算任务**：计算 3^22\n   - 结果：31,381,059,609'